# Benchmark Dynamical Localization in Floquet System

Dynamical localization is a characteristic quantum effect in periodically driven chaotic systems. In classical maps, namely discrete-time dynamical systems of the form $x_t \rightarrow x_{t+1}$, an initially localized distribution typically spreads over time in a diffusive manner. In their quantized counterparts, known as quantum maps, interference can suppress this spreading and keep the wavepacket concentrated around its initial momentum. Because this genuinely quantum effect relies on the coherent accumulation of phase over several time steps, while still admitting a relatively simple circuit implementation, it has been proposed as a clear benchmarking task for quantum hardware [[1]](#Pizzamiglio_etal). In particular, here we consider the quantum sawtooth map.

***

## The Model

We benchmark dynamical localization using the quantum sawtooth map, whose unitary evolution in momentum space after $T$ time steps is given by
$$
|\psi_T\rangle
=
\left(
\exp\left[-i\frac{2\pi}{N}\frac{\hat{p}^2}{2}\right]
\exp\left[iK\frac{2\pi}{N}\frac{\hat{q}^2}{2}\right]
\right)^T
|\psi_0\rangle.
$$

The model initializes the momentum register in a localized basis state,
$$
|\psi(0)\rangle = |-2\rangle,
$$
and evolves it for different final times $T$. To probe whether the state remains localized, we evaluate the output distribution after two and three kicks, namely at $T=2$ and $T=3$, and examine how much probability remains on the initial momentum value $|-2\rangle$.

## The Scoring Function

The scoring function is designed to quantify the persistence of the localization peak over time. Let
$$
P_T(-2)
$$
denote the measured probability of observing the momentum value $|-2\rangle$ after $T$ kicks. In a localized regime, this peak should remain significantly larger than the uniform background at both times. In contrast, delocalization, noise, or control errors tend to reduce the peak as probability spreads across the momentum register.

For an $n$-qubit register, the uniform baseline is $1/N$, where
$$
N = 2^n.
$$
We therefore define the normalized peak strength at time $T$ by
$$
\tilde{P}_T
=
\frac{P_T(-2)-1/N}{1-1/N},
$$
with values clipped to the interval $[0,1]$.

The final localization score is then defined as the geometric mean of the normalized peak strengths at the two sampled times,
$$
S = \sqrt{\tilde{P}_2 \tilde{P}_3}.
$$

This score rewards both the presence of a strong localization peak and its persistence over successive kicks. A value $S=0$ corresponds to a peak no better than the uniform background, while larger values indicate stronger and more stable localization around the initial momentum.

Since the system is finite, localization is not expected to produce a perfectly stationary or monotonic peak. Instead, the probability distribution may remain concentrated near the initial momentum while still exhibiting small oscillations or partial revivals. Accordingly, the score evaluates the strength of the localization peak at two successive times, rather than enforcing monotonic decay or exact time-independence. Because this is an absolute localization score for a finite system, even an ideal noiseless simulation need not attain the maximal value $S=1$.

***
***

In [1]:
import asyncio
import datetime
import sys

sys.path.insert(0, "..")

from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [2]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project
# reset_benchmark_project()
# ============================================================

In [3]:
from examples.dynamical_localization import QSMExample

***
***
## Benchmarking Dynamical Localization on 3-Qubits

Define a specific example on 3 qubits (the size is fixed for this example)

In [4]:
example = QSMExample()
example.show()

Quantum program link: https://platform.classiq.io/circuit/3LgmHW4xieZC9AF65RJls3gdA8a


https://platform.classiq.io/circuit/3LgmHW4xieZC9AF65RJls3gdA8a?login=True&version=24

### Set a `ResultCollector` with a File Name Fixed for the Current Example

In [5]:
FILENAME = example.default_results_filename

In [6]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [7]:
# Uncomment to clear the data of a previous run
#
# await collector.reset_file()

### Choose on Which Backend to Run 

We can print the list of backends:

In [8]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1, Emerald, IBEX Q1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1
C12: Callisto Emulator


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [9]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [10]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [11]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [12]:
print(
    "=" * 10
    + f"  {example.name}-{example.problem_size} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  localization-3 (2026-11-05 15:06:49.754830)   ==========


In [13]:
await collector.print_status()

========== (2026-11-05 15:06:49.761410)   ==========
localization-3 | Classiq - simulator | COMPLETED | score=0.922 | time=0.014 min
localization-3 | Classiq - simulator_statevector | COMPLETED | score=0.928 | time=0.014 min


## Refrences

<a id='Pizzamiglio_etal'>[1]</a>: [Andrea Pizzamiglio, Su Yeon Chang, Maria Bondani, Simone Montangero, Dario Gerace, Giuliano Benenti. "Dynamical localization simulated on actual quantum hardware." Entropy 23, 654 (2021).](https://arxiv.org/abs/2105.10813)